### This notebook is designed to help you revise essential problem-solving techniques, concepts, and functions every data scientist should know. Each task comes with a worked solution so you can compare your approach, understand the logic step by step, and strengthen the core Python skills needed for data science.

## Common leetcode problems

## 1. Customer Loss Ratios
- Task:
-Given a list of insurance records with fields: customer_id, premium, and claim_amount, write a function that:
•	Aggregates by customer_id
•	Computes:
o	total_premium
o	total_claims
o	loss_ratio = total_claims / total_premium (handle divide-by-zero)
•	Returns a list of customers sorted by descending loss_ratio, then by customer_id.
What it tests:
Dictionaries, grouping/aggregation, sorting with custom keys, defensive coding.
________________________________________


In [5]:
# Test dataset: list of dicts
records1 = [
    # Customer 101: moderate loss ratio (claims < premium)
    {"customer_id": 101, "premium": 1000.0, "claim_amount": 200.0},
    {"customer_id": 101, "premium": 500.0,  "claim_amount": 100.0},

    # Customer 202: high loss ratio (claims > premium)
    {"customer_id": 202, "premium": 800.0,  "claim_amount": 900.0},
    {"customer_id": 202, "premium": 200.0,  "claim_amount": 400.0},

    # Customer 303: no claims (loss ratio = 0)
    {"customer_id": 303, "premium": 1200.0, "claim_amount": 0.0},
    {"customer_id": 303, "premium": 300.0,  "claim_amount": 0.0},

    # Customer 404: zero premium but has claims (tests divide-by-zero handling)
    {"customer_id": 404, "premium": 0.0,    "claim_amount": 500.0},

    # Customer 505: same loss ratio as 101 to test secondary sort by customer_id
    {"customer_id": 505, "premium": 1500.0, "claim_amount": 300.0},
]


In [6]:
records1 

[{'customer_id': 101, 'premium': 1000.0, 'claim_amount': 200.0},
 {'customer_id': 101, 'premium': 500.0, 'claim_amount': 100.0},
 {'customer_id': 202, 'premium': 800.0, 'claim_amount': 900.0},
 {'customer_id': 202, 'premium': 200.0, 'claim_amount': 400.0},
 {'customer_id': 303, 'premium': 1200.0, 'claim_amount': 0.0},
 {'customer_id': 303, 'premium': 300.0, 'claim_amount': 0.0},
 {'customer_id': 404, 'premium': 0.0, 'claim_amount': 500.0},
 {'customer_id': 505, 'premium': 1500.0, 'claim_amount': 300.0}]

In [4]:
#Customer Loss Ratios
def compute_customer_loss_ratios(records):
    """
    records: list of dicts with keys: 'customer_id', 'premium', 'claim_amount'
    returns: list of dicts sorted by descending loss_ratio, then customer_id
    """
    # Create an empty dictionary to store aggregated results per customer.
    # Key: customer_id, Value: another dict with total_premium and total_claims.
    aggregates = {}

    # Loop through each insurance record in the input list.
    for record in records:
        # Extract the customer_id from the current record.
        customer_id = record["customer_id"]
        # Extract the premium (assume it's numeric).
        premium = record["premium"]
        # Extract the claim_amount (assume it's numeric).
        claim = record["claim_amount"]

        # If we haven't seen this customer_id before, initialize their totals.
        if customer_id not in aggregates:
            # Create an entry for this customer with starting totals at 0.
            aggregates[customer_id] = {
                "total_premium": 0.0,
                "total_claims": 0.0,
            }

        # Add this record's premium to the customer's total_premium.
        aggregates[customer_id]["total_premium"] += premium
        # Add this record's claim_amount to the customer's total_claims.
        aggregates[customer_id]["total_claims"] += claim

    # Now we convert the aggregated dictionary into a list of results.
    result = []

    # Loop through each customer_id and their aggregated values.
    for customer_id, totals in aggregates.items():
        # Get the total premium for this customer.
        total_premium = totals["total_premium"]
        # Get the total claims for this customer.
        total_claims = totals["total_claims"]

        # Handle divide-by-zero: if total_premium is 0, define loss_ratio as 0.0.
        if total_premium == 0:
            loss_ratio = 0.0
        else:
            # Otherwise, compute the loss_ratio = total_claims / total_premium.
            loss_ratio = total_claims / total_premium

        # Build a dictionary representing this customer's summary.
        result.append({
            "customer_id": customer_id,
            "total_premium": total_premium,
            "total_claims": total_claims,
            "loss_ratio": loss_ratio,
        })

    # Sort the result list:
    # - First by loss_ratio in descending order (hence -x["loss_ratio"])
    # - Then by customer_id in ascending order to break ties.
    result_sorted = sorted(
        result,
        key=lambda x: (-x["loss_ratio"], x["customer_id"])
    )

    # Return the sorted list.
    return result_sorted


In [7]:
compute_customer_loss_ratios(records1)

[{'customer_id': 202,
  'total_premium': 1000.0,
  'total_claims': 1300.0,
  'loss_ratio': 1.3},
 {'customer_id': 101,
  'total_premium': 1500.0,
  'total_claims': 300.0,
  'loss_ratio': 0.2},
 {'customer_id': 505,
  'total_premium': 1500.0,
  'total_claims': 300.0,
  'loss_ratio': 0.2},
 {'customer_id': 303,
  'total_premium': 1500.0,
  'total_claims': 0.0,
  'loss_ratio': 0.0},
 {'customer_id': 404,
  'total_premium': 0.0,
  'total_claims': 500.0,
  'loss_ratio': 0.0}]

In [8]:
def count_suspicious_transactions(transactions, allowed_countries_for_online):
    """
    transactions: list of dicts with keys:
        'amount' (float), 'country' (str),
        'is_international' (bool), 'channel' (str)
    allowed_countries_for_online: a set or list of allowed country codes/names
    returns: integer count of suspicious transactions
    """
    # Convert allowed_countries_for_online to a set for faster lookups (if it isn't already).
    allowed_set = set(allowed_countries_for_online)

    # Initialize a counter for suspicious transactions.
    suspicious_count = 0

    # Loop through each transaction in the input list.
    for tx in transactions:
        # Extract the relevant fields from the transaction dictionary.
        amount = tx["amount"]
        country = tx["country"]
        is_international = tx["is_international"]
        channel = tx["channel"]

        # Apply the three fraud rules using boolean logic.
        rule1 = amount > 5000
        rule2 = is_international and amount > 1000
        rule3 = (channel == "online") and (country not in allowed_set)

        # If any of the rules are True, the transaction is suspicious.
        if rule1 or rule2 or rule3:
            # Increment the suspicious transaction counter.
            suspicious_count += 1

    # Return the total number of suspicious transactions.
    return suspicious_count


In [11]:
allowed_countries_for_online = {"UK", "USA", "Germany"}

In [9]:
#data
transactions = [
    # 1) Suspicious by rule1: amount > 5000
    {
        "amount": 6000.0,
        "country": "UK",
        "is_international": False,
        "channel": "pos",
    },
    # 2) Suspicious by rule2: is_international == True and amount > 1000
    {
        "amount": 1500.0,
        "country": "France",
        "is_international": True,
        "channel": "pos",
    },
    # 3) Suspicious by rule3: channel == "online" and country NOT in allowed list
    {
        "amount": 800.0,
        "country": "Brazil",
        "is_international": True,
        "channel": "online",
    },
    # 4) Not suspicious: small amount, online, but country is allowed
    {
        "amount": 200.0,
        "country": "USA",
        "is_international": False,
        "channel": "online",
    },
    # 5) Suspicious: triggers rule1 and rule2 (still counts as 1)
    {
        "amount": 5200.0,
        "country": "Germany",
        "is_international": True,
        "channel": "online",
    },
    # 6) Not suspicious: low amount, domestic, atm
    {
        "amount": 900.0,
        "country": "India",
        "is_international": False,
        "channel": "atm",
    },
    # 7) Suspicious by rule3: online + country not allowed
    {
        "amount": 1200.0,
        "country": "Spain",
        "is_international": False,
        "channel": "online",
    },
    # 8) Suspicious by rule2: international + amount > 1000
    {
        "amount": 3000.0,
        "country": "USA",
        "is_international": True,
        "channel": "pos",
    },
]

In [12]:
count_suspicious_transactions(transactions, allowed_countries_for_online)

6

In [ ]:
# 3. Moving Average of Claim Amounts (O(n) sliding window)
def moving_average_claims(amounts, k):
    """
    amounts: list of daily claim amounts [a1, a2, ..., an]
    k: window size (integer)
    returns: list of k-day moving averages rounded to 2 decimals
    """
    # Get the number of data points.
    n = len(amounts)

    # If k is invalid (<=0 or larger than n), return an empty list.
    if k <= 0 or k > n:
        return []

    # Compute the sum of the first window of size k.
    window_sum = sum(amounts[0:k])
    # Initialize the result list with the average of the first window.
    result = [round(window_sum / k, 2)]

    # Slide the window from index 1 to index n-k.
    for i in range(1, n - k + 1):
        # Subtract the element that is leaving the window (amounts[i-1]).
        window_sum -= amounts[i - 1]
        # Add the new element entering the window (amounts[i + k - 1]).
        window_sum += amounts[i + k - 1]
        # Compute the current window's average and round to 2 decimals.
        avg = round(window_sum / k, 2)
        # Append it to the result list.
        result.append(avg)

    # Return the list of moving averages.
    return result
________________________________________


#### Task 1
##### Write a function to join the list of words below 
["Let's ", 'do ', 'this!']

In [ ]:
def combine(words):
    """ Join words together into a sentence 

    words : list of words to join 
    sentence : variables that stores the joined words

    """
    sentence  = " ".join(words)
    return sentence 



In [2]:
combine(["Let's ", 'do ', 'this!'])

"Let's  do  this!"

#### Task 2 
##### Split the words below into words and capitalize them 
"Let's  do  this today and tomorrow until we are fulfilled!"

In [5]:
def split_capitalize(sentence):
    """ This function splits the sentence into words and capitalizes it

    sentence : string of words to split and capitalize
    words: list of words separated """
    words = sentence.upper().split(" ")
    return words

In [6]:
split_capitalize("Let's do this today and tomorrow until we are fulfilled!")

["LET'S",
 'DO',
 'THIS',
 'TODAY',
 'AND',
 'TOMORROW',
 'UNTIL',
 'WE',
 'ARE',
 'FULFILLED!']

##### Task 3
- Given a list of lowercase strings, return a list with the strings in sorted in alphabetical order, except group all the strings that begin with `'x'` at the beginning of the list.<br>
e.g. ['mix', 'xyz', 'apple', 'xanadu', 'aardvark'] yields<br>
`['xanadu', 'xyz', 'aardvark', 'apple', 'mix']`<br>

Hint: this can be done by making 2 lists and sorting each of them before combining them.

Hint: Remember that Python's `list` object has a `sort()` function. Python also has a built-in function called `sorted()`. 

In [21]:
def x_sort(li_words):
    
    x_list = []
    non_x_list = []
    for words in li_words:
        if words[0] =='x':
            x_list.append(words)
        else:
            non_x_list.append(words)
    x_list = sorted(x_list)
    non_x_list = sorted(non_x_list)

    return x_list + non_x_list

        

In [22]:
x_sort(['xanadu', 'xyz', 'aardvark', 'apple', 'mix'])

['xanadu', 'xyz', 'aardvark', 'apple', 'mix']

##### Task 4
- Proteins are the building blocks of life. The process starts when another protein (a polymerase) reads your genetic code and performs transcription, the code (RNA) for making protein. Ribosomes will then attach to this RNA code to begin making specific proteins. Your genetic code is made up of pairs of nucleotides. Once transcribed into the code, these nucleotides contain the following nucleotides:

U,G,A,C.

UGA, UAA, and UAG are stop codons and AUG is a start codon so that the protein ,ribosome, can identify where to stop and start the protein formation.

A genetics lab has tasked you to design a function which recognises the length of the RNA code (excluding the start and stop codons). The function must also determine if there code is actually a valid piece of code (must contain AUG as a start codon and UGA, UAA, or UAG at the end).

Note: RNA starts with a start codon AUG and ends with UGA, UAA, or UAG

The function must return "Not readable RNA code" if the above conditions are not met.

In [54]:
def rna_length(RNA):
    if RNA[0:3] =='AUG' and RNA[-3:] in ('UGA','UAA','UAG'):
        RNA =RNA[3:]
        RNA = RNA[0:-3]
        rna_len =len(RNA)
        return rna_len    
    else:
        return "Not readable RNA code"

In [45]:
rna_length('AUGUAGCAUAA')

5

In [47]:
rna_length('AUGUAGCAUAA')==5

True

In [49]:
rna_length('AUGUUAUAG') == 3

True

In [50]:
rna_length('AUGUAGGCACAUUUAUGCUCCUGA') == 18

True

In [55]:
rna_length('AUGAGGCACCUUCUGCUCCUUAC') == "Not readable RNA code"

True

## Task 4

**Finding the Sum of a Symmetrical Sub-List**

**Task:** Create a function called symmetrical_sum which takes a Python list of integers as input and searches for a 'symmetrical' inner-portion of the list. Here symmetry is said to occur if the value of the  ith  element from the start of the list is equal to the value of the  ith  value from the end of the list. The inner-portion then consists of all elements between and including these equal values. If found, the function returns both the symmetrical part of the list, as well as the sum of its constituent elements as a tuple.

Function arguments: lst - a Python list containing elements of type int. Note: len(lst)  ≥0 
Returns: A Python tuple of the form ([symmetrical-portion], sum-of-symmetrical-portion)

**Example:** symmetrical_sum([10,8,7,5,9,8,15]) == ([8, 7, 5, 9, 8], 37). Here a symmetrical portion of the list is formed by the elements at the second and second-last index, which share a value of 8. The function returns this symmetrical portion, as well as its sum of 37 within a a single tuple.

In [ ]:
def symmetrical_sum(lnt):
    

Flag high-value transactions

Task
Given a list of transactions, create:

a column is_high_value (True if amount > 1000) using lambda

filter the DataFrame to show only high-value transactions.

In [ ]:
import pandas as pd

# Sample data
df = pd.DataFrame({
    "transaction_id": [1, 2, 3, 4],
    "amount": [150.0, 2300.5, 999.9, 5000.0]
})

# Create a boolean flag using lambda + apply
df["is_high_value"] = df["amount"].apply(lambda x: x > 1000)

# Filter to only high-value transactions
high_value_df = df[df["is_high_value"]]

print(df)
print("\nHigh value only:")
print(high_value_df)


Map gender codes to full text & filter missing

Task

Convert M/F in a gender column to "Male" / "Female" using map.

Use a lambda to replace unknown codes with "Unknown".

Filter rows where gender_full != "Unknown".

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "name": ["Alex", "Bea", "Chris", "Dana"],
    "gender": ["M", "F", "X", None]
})

gender_map = {"M": "Male", "F": "Female"}

# First map known values
df["gender_full"] = df["gender"].map(gender_map)

# Use lambda to fill missing/unknown codes
df["gender_full"] = df["gender_full"].apply(lambda x: x if pd.notnull(x) else "Unknown")

# Filter out unknowns
known_gender_df = df[df["gender_full"] != "Unknown"]

print(df)
print("\nOnly known genders:")
print(known_gender_df)


3️⃣ Compute line revenue and filter top sellers

Task

For each row, compute revenue = price * quantity using lambda with apply(axis=1).

Filter products with revenue >= 1000.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "product": ["A", "B", "C", "D"],
    "price": [10, 50, 100, 200],
    "quantity": [20, 5, 15, 3]
})

# Row-wise lambda
df["revenue"] = df.apply(lambda row: row["price"] * row["quantity"], axis=1)

# Filter high revenue
top_sellers = df[df["revenue"] >= 1000]

print(df)
print("\nTop sellers (revenue >= 1000):")
print(top_sellers)


4️⃣ Extract year from date and filter a specific year

Task

Convert date strings to datetime.

Use lambda to create a year column.

Filter to only rows from 2024.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "order_id": [101, 102, 103, 104],
    "order_date": ["2023-12-31", "2024-01-10", "2024-05-01", "2025-02-20"]
})

df["order_date"] = pd.to_datetime(df["order_date"])

# Extract year with lambda
df["year"] = df["order_date"].apply(lambda d: d.year)

# Filter for 2024 orders
orders_2024 = df[df["year"] == 2024]

print(df)
print("\nOrders in 2024:")
print(orders_2024)


lean product names into “slugs” and filter by keyword

Task

Use lambda to create a URL-friendly slug from product_name
(lowercase, spaces → hyphens).

Filter products whose name contains the word "Pro" (case-insensitive).

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "product_name": ["Data Pro Course", "Intro to Python", "ML Pro Bootcamp", "Excel Basics"]
})

# Create slug: lowercase and replace spaces with '-'
df["slug"] = df["product_name"].apply(
    lambda s: s.lower().replace(" ", "-")
)

# Filter names containing "pro" (case-insensitive) using lambda
pro_products = df[df["product_name"].apply(lambda s: "pro" in s.lower())]

print(df)
print("\nProducts containing 'Pro':")
print(pro_products)
